In [1]:
# !pip install -q openpyxl tqdm

from openpyxl import load_workbook
from tqdm.auto import tqdm
import re
import html
import unicodedata

# =========================
# 1. File paths
# =========================

# Use your latest cleaned file here.
EXPORT_FILE = "/content/franckenachlass_5.xls_Wed_Sep_09_07_12_33_2026part_1.xlsx"

# Corrected VIAF file
PERSON_FILE = "/content/VIAF_updated.xlsx"

OUTPUT_FILE = "/content/franckenachlass_6.xls_Wed_Sep_09_07_12_33_2026part_1.xlsx"

HEADER_ROW = 1

#linked columns
CREATOR_LINKED_NEW = "creator_linked_new"
RECEIVER_LINKED_NEW = "receiver_linked_new"
SENDER_PLACE_LINKED_NEW = "sender_place_linked_new"
RECEIVER_PLACE_LINKED_NEW = "receiver_place_linked_new"

FALLBACK_ORIGINAL_LINKED_COLS = {
    CREATOR_LINKED_NEW: "creator_linked",
    RECEIVER_LINKED_NEW: "receiver_linked",
    SENDER_PLACE_LINKED_NEW: "sender_place_linked",
    RECEIVER_PLACE_LINKED_NEW: "receiver_place_linked",
}

RECEIVER_COL = "receiver"

# Corrected VIAF file columns
PERSON_NAME_COL_CANDIDATES = [
    "personStdName_original",
    "personStdName"
]

OLD_VIAF_COL_CANDIDATES = [
    "LODVIAF_original",
    "LODVIAF",
    "LODVIAF_inDB"
]

NEW_VIAF_URI_COL_CANDIDATES = [
    "VIAF_extractedfrWikidataentry_uri",
    "VIAF_inWikidataentry_uri",
    "VIAF_uri"
]

NEW_VIAF_LABEL_COL_CANDIDATES = [
    "VIAF_extractedfrWikidataentry_label",
    "VIAF_inWikidataentry_label",
    "VIAF_label"
]


# =========================
# 2. Helper functions
# =========================

def get_header_positions(ws):
    positions = {}
    for col in range(1, ws.max_column + 1):
        value = ws.cell(HEADER_ROW, col).value
        if value is not None:
            header = str(value).strip()
            positions.setdefault(header, []).append(col)
    return positions


def get_first_col(header_positions, col_name):
    if col_name not in header_positions:
        raise ValueError(f"Missing required column: {col_name}")
    return header_positions[col_name][0]


def get_first_available_col(header_positions, candidates, required=True):
    for col_name in candidates:
        if col_name in header_positions:
            return col_name, header_positions[col_name][0]

    if required:
        raise ValueError(f"Missing all candidate columns: {candidates}")

    return None, None


def is_blank(value):
    return value is None or str(value).strip() == ""


def normalize_space(text):
    if text is None:
        return ""
    return re.sub(r"\s+", " ", str(text)).strip()


def normalize_text_for_match(text):
    if text is None:
        return ""

    text = str(text).strip().strip("[]")

    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))

    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def clean_person_name(name):
    if name is None:
        return ""

    name = str(name).strip()
    name = name.strip("[]")
    name = normalize_space(name)
    return name


def strip_life_dates_for_match(name):
    if name is None:
        return ""

    name = str(name)

    name = re.sub(
        r"\s*\([^)]*(?:\d{3,4}|b\.|d\.|fl\.|ca\.|c\.)[^)]*\)",
        "",
        name,
        flags=re.IGNORECASE
    )

    name = re.sub(
        r"\s*,\s*-?\d{3,4}\s*(?:[-–]\s*\d{0,4})?",
        "",
        name
    )

    name = re.sub(r"\b\d{3,4}\s*[-–]\s*\d{0,4}\b", "", name)

    name = re.sub(
        r"\b(b|d|fl|ca|c)\.\s*\d{3,4}\b",
        "",
        name,
        flags=re.IGNORECASE
    )

    name = re.sub(r"\b\d{3,4}\b", "", name)

    name = re.sub(r"\s+", " ", name).strip()
    name = re.sub(r"\s+,", ",", name).strip()
    name = re.sub(r",\s*$", "", name).strip()

    return name


def person_std_to_full_name(person_std):
    person_std = clean_person_name(person_std)
    person_std = strip_life_dates_for_match(person_std)

    if "," not in person_std:
        return person_std

    last, first = person_std.split(",", 1)
    last = normalize_space(last)
    first = normalize_space(first)

    if first and last:
        return f"{first} {last}"

    return person_std


def make_author_std_name(fname, mname, lname):
    fname = normalize_space(fname)
    mname = normalize_space(mname)
    lname = normalize_space(lname)

    first_parts = []

    if fname:
        first_parts.append(fname)

    if mname:
        first_parts.append(mname)

    first_middle = normalize_space(" ".join(first_parts))

    if lname and first_middle:
        return f"{lname}, {first_middle}"

    if lname:
        return lname

    if first_middle:
        return first_middle

    return ""


def extract_viaf_id(value):
    if value is None:
        return None

    text = str(value).strip()

    if text == "":
        return None

    if re.fullmatch(r"\d+", text):
        return text

    m = re.search(r"viaf\.org/(?:en/)?viaf/(\d+)", text, flags=re.IGNORECASE)
    if m:
        return m.group(1)

    m = re.search(r"\b(\d{3,})\b", text)
    if m:
        return m.group(1)

    return None


def normalize_viaf_uri(uri):
    viaf_id = extract_viaf_id(uri)

    if viaf_id:
        return f"https://viaf.org/viaf/{viaf_id}"

    if uri is None:
        return ""

    return str(uri).strip().rstrip("/")


def build_viaf_anchor(uri, label):
    uri = html.escape(normalize_viaf_uri(uri), quote=True)
    label = html.escape(str(label).strip(), quote=False)
    return f'<p><a href="{uri}">{label}</a> [VIAF]</p>'


def append_viaf_anchor_if_missing(cell_value, anchor_html, new_viaf_id):
    if is_blank(cell_value):
        return anchor_html, True

    cell_text = str(cell_value)

    existing_ids = set()

    for m in VIAF_ANCHOR_RE.finditer(cell_text):
        existing_id = extract_viaf_id(m.group("href"))
        if existing_id:
            existing_ids.add(existing_id)

    if new_viaf_id in existing_ids:
        return cell_text, False

    return cell_text + anchor_html, True


def get_or_create_target_col(ws, header_positions, new_col_name):
    if new_col_name in header_positions:
        return header_positions[new_col_name][0]

    fallback_col_name = FALLBACK_ORIGINAL_LINKED_COLS[new_col_name]

    if fallback_col_name not in header_positions:
        raise ValueError(
            f"Missing both '{new_col_name}' and fallback '{fallback_col_name}'"
        )

    fallback_idx = header_positions[fallback_col_name][0]

    new_idx = ws.max_column + 1
    ws.cell(HEADER_ROW, new_idx).value = new_col_name

    for row in range(HEADER_ROW + 1, ws.max_row + 1):
        ws.cell(row, new_idx).value = ws.cell(row, fallback_idx).value

    header_positions[new_col_name] = [new_idx]

    print(f"Created '{new_col_name}' from '{fallback_col_name}'")

    return new_idx


def find_author_groups(header_positions):
    author_nums = set()

    for header in header_positions:
        m = re.fullmatch(r"author(\d+)_(?:fname|mname|lname)", header)
        if m:
            author_nums.add(int(m.group(1)))

    groups = []

    for n in sorted(author_nums):
        fname_col = f"author{n}_fname"
        mname_col = f"author{n}_mname"
        lname_col = f"author{n}_lname"

        if fname_col in header_positions or lname_col in header_positions:
            groups.append({
                "author_num": n,
                "fname_idx": header_positions[fname_col][0] if fname_col in header_positions else None,
                "mname_idx": header_positions[mname_col][0] if mname_col in header_positions else None,
                "lname_idx": header_positions[lname_col][0] if lname_col in header_positions else None,
            })

    return groups


def add_to_multimap(mapping, key, record):
    if not key:
        return
    mapping.setdefault(key, []).append(record)


# =========================
# 3. VIAF anchor regex
# =========================

VIAF_ANCHOR_RE = re.compile(
    r'(?P<start><a\b[^>]*\bhref=["\']'
    r'(?P<href>https?://(?:www\.)?viaf\.org/(?:en/)?viaf/(?P<viafid>\d+)[^"\']*)'
    r'["\'][^>]*>)'
    r'(?P<label>.*?)'
    r'(?P<end></a>)',
    re.IGNORECASE | re.DOTALL
)


# =========================
# 4. Load corrected VIAF file
# =========================

person_wb = load_workbook(PERSON_FILE, data_only=True)

person_ws = None
person_headers = None

for ws in person_wb.worksheets:
    headers = get_header_positions(ws)

    has_person_col = any(c in headers for c in PERSON_NAME_COL_CANDIDATES)
    has_old_viaf_col = any(c in headers for c in OLD_VIAF_COL_CANDIDATES)
    has_new_uri_col = any(c in headers for c in NEW_VIAF_URI_COL_CANDIDATES)
    has_new_label_col = any(c in headers for c in NEW_VIAF_LABEL_COL_CANDIDATES)

    if has_person_col and has_old_viaf_col and has_new_uri_col and has_new_label_col:
        person_ws = ws
        person_headers = headers
        break

if person_ws is None:
    raise ValueError("Could not find a corrected VIAF sheet with required columns.")

print("Using corrected VIAF sheet:", person_ws.title)

person_name_col_used, person_name_idx = get_first_available_col(
    person_headers,
    PERSON_NAME_COL_CANDIDATES
)

old_viaf_col_used, old_viaf_idx = get_first_available_col(
    person_headers,
    OLD_VIAF_COL_CANDIDATES
)

new_viaf_uri_col_used, new_viaf_uri_idx = get_first_available_col(
    person_headers,
    NEW_VIAF_URI_COL_CANDIDATES
)

new_viaf_label_col_used, new_viaf_label_idx = get_first_available_col(
    person_headers,
    NEW_VIAF_LABEL_COL_CANDIDATES
)

print("Using person name column:", person_name_col_used)
print("Using old VIAF column:", old_viaf_col_used)
print("Using new VIAF URI column:", new_viaf_uri_col_used)
print("Using new VIAF label column:", new_viaf_label_col_used)

replace_by_old_viaf_id = {}
replace_by_new_viaf_id = {}   # NEW: allows matching existing VIAF IDs that are already corrected

insert_by_creator_key = {}
insert_by_receiver_key = {}

person_records = []
duplicate_old_viaf_ids = []

for row in range(HEADER_ROW + 1, person_ws.max_row + 1):
    person_name_raw = person_ws.cell(row, person_name_idx).value
    old_viaf_raw = person_ws.cell(row, old_viaf_idx).value
    new_uri_raw = person_ws.cell(row, new_viaf_uri_idx).value
    new_label_raw = person_ws.cell(row, new_viaf_label_idx).value

    if is_blank(new_uri_raw) or is_blank(new_label_raw):
        continue

    if not is_blank(person_name_raw):
        person_std = clean_person_name(person_name_raw)
    else:
        person_std = strip_life_dates_for_match(str(new_label_raw).strip())

    person_full = person_std_to_full_name(person_std)

    old_viaf_id = extract_viaf_id(old_viaf_raw)
    new_viaf_id = extract_viaf_id(new_uri_raw)

    if not new_viaf_id:
        continue

    record = {
        "person_file_row": row,
        "personStdName_original": person_std,
        "person_full_name": person_full,
        "old_viaf_original": old_viaf_raw,
        "old_viaf_id": old_viaf_id,
        "new_viaf_uri": normalize_viaf_uri(new_uri_raw),
        "new_viaf_id": new_viaf_id,
        "new_viaf_label": str(new_label_raw).strip(),
    }

    person_records.append(record)

    # Replacement map using old VIAF ID
    if old_viaf_id:
        if old_viaf_id in replace_by_old_viaf_id:
            duplicate_old_viaf_ids.append(old_viaf_id)
        replace_by_old_viaf_id[old_viaf_id] = record

    # NEW: Replacement map using new/corrected VIAF ID
    # This catches cases where the existing workbook already has the corrected VIAF ID,
    # but the label still needs to be updated.
    if new_viaf_id:
        replace_by_new_viaf_id[new_viaf_id] = record

    # Insertion maps: every record is eligible
    creator_name_clean = strip_life_dates_for_match(person_std)
    receiver_full_clean = strip_life_dates_for_match(person_full)

    add_to_multimap(
        insert_by_creator_key,
        normalize_text_for_match(creator_name_clean),
        record
    )

    add_to_multimap(
        insert_by_receiver_key,
        normalize_text_for_match(receiver_full_clean),
        record
    )

    add_to_multimap(
        insert_by_receiver_key,
        normalize_text_for_match(creator_name_clean),
        record
    )

print("Loaded corrected VIAF records:", len(person_records))
print("Records for replacing old VIAF:", len(replace_by_old_viaf_id))
print("Creator insertion keys:", len(insert_by_creator_key))
print("Receiver insertion keys:", len(insert_by_receiver_key))
print("Duplicate old VIAF IDs:", len(set(duplicate_old_viaf_ids)))


# =========================
# 5. Load export file
# =========================

export_wb = load_workbook(EXPORT_FILE)

export_ws = None
export_headers = None

for ws in export_wb.worksheets:
    headers = get_header_positions(ws)

    has_linked_cols = (
        CREATOR_LINKED_NEW in headers
        or FALLBACK_ORIGINAL_LINKED_COLS[CREATOR_LINKED_NEW] in headers
    )

    has_receiver = RECEIVER_COL in headers

    has_author = any(
        re.fullmatch(r"author\d+_(?:fname|mname|lname)", h) for h in headers
    )

    if has_linked_cols and has_receiver and has_author:
        export_ws = ws
        export_headers = headers
        break

if export_ws is None:
    raise ValueError("Could not find export sheet with linked, receiver, and author columns.")

print("Using export sheet:", export_ws.title)

creator_linked_new_idx = get_or_create_target_col(
    export_ws,
    export_headers,
    CREATOR_LINKED_NEW
)

receiver_linked_new_idx = get_or_create_target_col(
    export_ws,
    export_headers,
    RECEIVER_LINKED_NEW
)

sender_place_linked_new_idx = get_or_create_target_col(
    export_ws,
    export_headers,
    SENDER_PLACE_LINKED_NEW
)

receiver_place_linked_new_idx = get_or_create_target_col(
    export_ws,
    export_headers,
    RECEIVER_PLACE_LINKED_NEW
)

receiver_idx = get_first_col(export_headers, RECEIVER_COL)

author_groups = find_author_groups(export_headers)

if not author_groups:
    raise ValueError("No authorN_fname/mname/lname groups found.")

print("Found author groups:")
for group in author_groups:
    n = group["author_num"]
    parts = []

    if group.get("fname_idx") is not None:
        parts.append(f"author{n}_fname")
    if group.get("mname_idx") is not None:
        parts.append(f"author{n}_mname")
    if group.get("lname_idx") is not None:
        parts.append(f"author{n}_lname")

    print("  " + " + ".join(parts))

target_replace_columns = [
    ("creator_linked_new", creator_linked_new_idx),
    ("receiver_linked_new", receiver_linked_new_idx),
    ("sender_place_linked_new", sender_place_linked_new_idx),
    ("receiver_place_linked_new", receiver_place_linked_new_idx),
]


# =========================
# 6. Replace existing VIAF anchors
# =========================

replace_log = []
existing_viaf_not_in_correction_log = []
used_record_rows = set()
total_replaced = 0


def replace_viaf_in_cell(cell_value, row_num, col_name):
    if cell_value is None or not isinstance(cell_value, str):
        return cell_value, []

    if "viaf.org" not in cell_value.lower():
        return cell_value, []

    changes = []

    def repl(match):
        old_href = match.group("href")
        old_label = match.group("label")
        old_viaf_id = extract_viaf_id(old_href)

        rec = (
            replace_by_old_viaf_id.get(old_viaf_id)
            or replace_by_new_viaf_id.get(old_viaf_id)
        )

        if rec is None:
            existing_viaf_not_in_correction_log.append([
                row_num,
                col_name,
                old_viaf_id,
                old_href,
                old_label
            ])
            return match.group(0)

            new_href = rec["new_viaf_uri"]
            new_label = rec["new_viaf_label"]

            # ---------------------------------------------------------
            # Do NOT replace if the URI difference is ONLY a trailing /
            # and the label is already the same.
            #
            # Example:
            # https://viaf.org/viaf/12345
            # https://viaf.org/viaf/12345/
            # are treated as equivalent here.
            # ---------------------------------------------------------
            old_href_compare = html.unescape(str(old_href)).strip().rstrip("/")
            new_href_compare = html.unescape(str(new_href)).strip().rstrip("/")

            old_label_compare = html.unescape(str(old_label)).strip()
            new_label_compare = html.unescape(str(new_label)).strip()

            if (
                old_href_compare == new_href_compare
                and old_label_compare == new_label_compare
            ):
                # Nothing meaningful changed.
                # Keep the original anchor exactly as it is.
                used_record_rows.add(rec["person_file_row"])
                return match.group(0)


            # Otherwise, actually replace because URI/ID or label changed
            escaped_new_href = html.escape(new_href, quote=True)
            escaped_new_label = html.escape(new_label, quote=False)

            changes.append([
                row_num,
                col_name,
                rec["person_file_row"],
                rec["personStdName_original"],
                old_viaf_id,
                old_href,
                old_label,
                new_href,
                new_label
            ])

            used_record_rows.add(rec["person_file_row"])

            return f'<a href="{escaped_new_href}">{escaped_new_label}</a>'

    new_value = VIAF_ANCHOR_RE.sub(repl, cell_value)

    return new_value, changes


for col_name, col_idx in target_replace_columns:
    for row in tqdm(
        range(HEADER_ROW + 1, export_ws.max_row + 1),
        desc=f"Replacing VIAF in {col_name}"
    ):
        cell = export_ws.cell(row, col_idx)
        old_value = cell.value

        new_value, changes = replace_viaf_in_cell(old_value, row, col_name)

        if changes:
            cell.value = new_value
            total_replaced += len(changes)
            replace_log.extend(changes)

print("Total existing VIAF anchors replaced:", total_replaced)


# =========================
# 7. Insert missing VIAF
# =========================

insert_log = []
ambiguous_insert_log = []
used_insert_record_rows = set()
total_inserted = 0


def insert_records_into_cell(row, target_col_idx, target_col_name, records, match_source):
    global total_inserted

    if not records:
        return

    unique = {}
    for rec in records:
        unique[rec["new_viaf_id"]] = rec

    records = list(unique.values())

    if len(records) > 1:
        ambiguous_insert_log.append([
            row,
            target_col_name,
            match_source,
            "; ".join(
                f'{rec["person_file_row"]}: {rec["personStdName_original"]} -> {rec["new_viaf_uri"]}'
                for rec in records
            )
        ])
        return

    rec = records[0]

    anchor = build_viaf_anchor(
        rec["new_viaf_uri"],
        rec["new_viaf_label"]
    )

    cell = export_ws.cell(row, target_col_idx)
    old_cell_value = cell.value

    new_cell_value, did_insert = append_viaf_anchor_if_missing(
        old_cell_value,
        anchor,
        rec["new_viaf_id"]
    )

    if did_insert:
        cell.value = new_cell_value
        total_inserted += 1
        used_record_rows.add(rec["person_file_row"])
        used_insert_record_rows.add(rec["person_file_row"])

        insert_log.append([
            row,
            target_col_name,
            match_source,
            rec["person_file_row"],
            rec["personStdName_original"],
            rec["person_full_name"],
            rec["old_viaf_original"],
            rec["old_viaf_id"],
            rec["new_viaf_id"],
            rec["new_viaf_uri"],
            rec["new_viaf_label"],
            old_cell_value,
            new_cell_value
        ])


for row in tqdm(
    range(HEADER_ROW + 1, export_ws.max_row + 1),
    desc="Inserting missing VIAF"
):
    # Creator insertion
    for group in author_groups:
        fname = (
            export_ws.cell(row, group["fname_idx"]).value
            if group.get("fname_idx") is not None
            else None
        )

        mname = (
            export_ws.cell(row, group["mname_idx"]).value
            if group.get("mname_idx") is not None
            else None
        )

        lname = (
            export_ws.cell(row, group["lname_idx"]).value
            if group.get("lname_idx") is not None
            else None
        )

        author_std = make_author_std_name(fname, mname, lname)

        if is_blank(author_std):
            continue

        author_key = normalize_text_for_match(
            strip_life_dates_for_match(author_std)
        )

        records = insert_by_creator_key.get(author_key, [])

        if records:
            insert_records_into_cell(
                row=row,
                target_col_idx=creator_linked_new_idx,
                target_col_name="creator_linked_new",
                records=records,
                match_source=f"author{group['author_num']}: {author_std}"
            )

    # Receiver insertion
    receiver_value = export_ws.cell(row, receiver_idx).value

    if not is_blank(receiver_value):
        receiver_text = normalize_space(receiver_value)

        possible_receiver_parts = [receiver_text]

        possible_receiver_parts.extend(
            [
                part.strip()
                for part in re.split(r"[;|\n/]+", receiver_text)
                if part.strip()
            ]
        )

        seen_receiver_parts = set()

        for receiver_part in possible_receiver_parts:
            key = normalize_text_for_match(
                strip_life_dates_for_match(receiver_part)
            )

            if not key or key in seen_receiver_parts:
                continue

            seen_receiver_parts.add(key)

            records = insert_by_receiver_key.get(key, [])

            if records:
                insert_records_into_cell(
                    row=row,
                    target_col_idx=receiver_linked_new_idx,
                    target_col_name="receiver_linked_new",
                    records=records,
                    match_source=f"receiver: {receiver_part}"
                )

print("Total new VIAF anchors inserted:", total_inserted)
print("Ambiguous insert cases skipped:", len(ambiguous_insert_log))


# =========================
# 8. Add log sheets
# =========================

log_sheet_names = [
    "viaf_replace_log",
    "viaf_insert_log",
    "viaf_ambig_log",
    "viaf_missing_in_term_list",
    "viaf_duplicate_ids",
    "viaf_not_applied"
]

for sheet_name in log_sheet_names:
    if sheet_name in export_wb.sheetnames:
        del export_wb[sheet_name]


ws_replace = export_wb.create_sheet("viaf_replace_log")
ws_replace.append([
    "export_row",
    "column",
    "person_file_row",
    "personStdName_original",
    "old_viaf_id",
    "old_viaf_href",
    "old_viaf_label",
    "new_viaf_href",
    "new_viaf_label"
])

for item in replace_log:
    ws_replace.append(item)


ws_insert = export_wb.create_sheet("viaf_insert_log")
ws_insert.append([
    "export_row",
    "target_column",
    "match_source",
    "person_file_row",
    "personStdName_original",
    "person_full_name",
    "old_viaf_original",
    "old_viaf_id",
    "new_viaf_id",
    "new_viaf_href",
    "new_viaf_label",
    "old_cell_value",
    "new_cell_value"
])

for item in insert_log:
    ws_insert.append(item)


ws_ambiguous = export_wb.create_sheet("viaf_ambig_log")
ws_ambiguous.append([
    "export_row",
    "target_column",
    "match_source",
    "possible_viaf_records"
])

for item in ambiguous_insert_log:
    ws_ambiguous.append(item)


ws_existing = export_wb.create_sheet("viaf_missing_in_term_list")
ws_existing.append([
    "export_row",
    "column",
    "existing_viaf_id",
    "existing_viaf_href",
    "existing_viaf_label"
])

seen_existing = set()

for item in existing_viaf_not_in_correction_log:
    key = tuple(item)
    if key in seen_existing:
        continue

    seen_existing.add(key)
    ws_existing.append(item)


ws_dupes = export_wb.create_sheet("viaf_duplicate_ids")
ws_dupes.append(["duplicate_old_viaf_id"])

for viaf_id in sorted(set(duplicate_old_viaf_ids)):
    ws_dupes.append([viaf_id])


ws_not_applied = export_wb.create_sheet("viaf_not_applied")
ws_not_applied.append([
    "person_file_row",
    "personStdName_original",
    "person_full_name",
    "old_viaf_original",
    "old_viaf_id",
    "new_viaf_id",
    "new_viaf_href",
    "new_viaf_label",
    "reason"
])

for rec in person_records:
    if rec["person_file_row"] not in used_record_rows:
        ws_not_applied.append([
            rec["person_file_row"],
            rec["personStdName_original"],
            rec["person_full_name"],
            rec["old_viaf_original"],
            rec["old_viaf_id"],
            rec["new_viaf_id"],
            rec["new_viaf_uri"],
            rec["new_viaf_label"],
            "Not replaced and not inserted. No safe creator/receiver name match, skipped as ambiguous, or not present in export."
        ])


for ws_log in [
    ws_replace,
    ws_insert,
    ws_ambiguous,
    ws_existing,
    ws_dupes,
    ws_not_applied
]:
    for col in range(1, ws_log.max_column + 1):
        ws_log.column_dimensions[ws_log.cell(1, col).column_letter].width = 34


# =========================
# 9. Save safely
# =========================

for ws in export_wb.worksheets:
    ws.sheet_state = "visible"

export_wb.active = export_wb.sheetnames.index(export_ws.title)

if export_wb.views:
    export_wb.views[0].activeTab = export_wb.sheetnames.index(export_ws.title)
    export_wb.views[0].firstSheet = 0

export_wb.save(OUTPUT_FILE)

print("Done.")
print("Saved file:", OUTPUT_FILE)

Using corrected VIAF sheet: Sheet1
Using person name column: personStdName_original
Using old VIAF column: LODVIAF_original
Using new VIAF URI column: VIAF_extractedfrWikidataentry_uri
Using new VIAF label column: VIAF_extractedfrWikidataentry_label
Loaded corrected VIAF records: 324
Records for replacing old VIAF: 87
Creator insertion keys: 322
Receiver insertion keys: 630
Duplicate old VIAF IDs: 1
Using export sheet: 40427936
Found author groups:
  author1_fname + author1_mname + author1_lname
  author2_fname + author2_mname + author2_lname
  author3_fname + author3_mname + author3_lname
  author4_fname + author4_mname + author4_lname
  author5_fname + author5_mname + author5_lname


Replacing VIAF in creator_linked_new:   0%|          | 0/324 [00:00<?, ?it/s]

Replacing VIAF in receiver_linked_new:   0%|          | 0/324 [00:00<?, ?it/s]

Replacing VIAF in sender_place_linked_new:   0%|          | 0/324 [00:00<?, ?it/s]

Replacing VIAF in receiver_place_linked_new:   0%|          | 0/324 [00:00<?, ?it/s]

Total existing VIAF anchors replaced: 0


Inserting missing VIAF:   0%|          | 0/324 [00:00<?, ?it/s]

Total new VIAF anchors inserted: 0
Ambiguous insert cases skipped: 0
Done.
Saved file: /content/franckenachlass_6.xls_Wed_Sep_09_07_12_33_2026part_1.xlsx
